In [0]:
from pyspark.sql.functions import current_timestamp, lit, sha2, concat_ws, col

In [0]:
%run ./00_config

In [0]:
dbutils.widgets.dropdown("environment", "dev", ["dev", "test", "prod"])
env = dbutils.widgets.get("environment")

config = get_config(env)
source_path = build_volume_path(config)
batch_id = config["batch_id"]

print_config(config)

In [0]:
def ingest_csv_to_bronze(file_name, target_table_name, source_system):
    file_path = f"{source_path}/{file_name}"
    target_table = build_table_name(config, "bronze_schema", target_table_name)

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(file_path)
    )

    source_columns = df.columns

    df_bronze = (
        df.withColumn("ingestion_datetime", current_timestamp())
          .withColumn("source_file_name", col("_metadata.file_path"))
          .withColumn("source_system", lit(source_system))
          .withColumn("batch_id", lit(batch_id))
          .withColumn(
              "record_hash",
              sha2(concat_ws("||", *[col(c).cast("string") for c in source_columns]), 256)
          )
    )

    df_bronze.write.format("delta").mode("overwrite").saveAsTable(target_table)

    print(f"Loaded {file_name} into {target_table}. Rows written: {df_bronze.count()}")

In [0]:
ingest_csv_to_bronze("policyholders.csv", "bronze_policyholders", "customer_system")
ingest_csv_to_bronze("policies.csv", "bronze_policies", "policy_admin_system")
ingest_csv_to_bronze("claims.csv", "bronze_claims", "claims_system")
ingest_csv_to_bronze("claim_payments.csv", "bronze_claim_payments", "payment_system")
ingest_csv_to_bronze("claim_reserves.csv", "bronze_claim_reserves", "reserve_system")
ingest_csv_to_bronze("fraud_indicators.csv", "bronze_fraud_indicators", "fraud_system")
ingest_csv_to_bronze("claim_events.csv", "bronze_claim_events", "claims_system")
ingest_csv_to_bronze("adjusters.csv", "bronze_adjusters", "hr_system")

In [0]:
spark.sql(f"SHOW TABLES IN {config['catalog_name']}.{config['bronze_schema']}").show(truncate=False)

In [0]:
display(spark.table(build_table_name(config, "bronze_schema", "bronze_claims")))